# 02 — Model tuning, calibration, and frozen planning coefficients

**Goal:** select CTR and clearing-price models without using the strict future holdout,
calibrate their outputs, and construct the compact planning table consumed by the MILP.


## Leakage contract

Hyperparameters are chosen inside the fit period. Probability and upper-cost calibration
use the next chronological block; validation chooses calibration behavior and reports
development diagnostics. Strict test is scored only after every choice is frozen.


In [ ]:
from pathlib import Path
import os

# Run correctly whether Jupyter starts in the project root or in notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
ARTIFACT_ROOT = Path(os.getenv("IPINYOU_ARTIFACT_ROOT", PROJECT_ROOT / "artifacts"))
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")


In [ ]:
from pathlib import Path
from itertools import product
import hashlib
import json
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:  # Allows the script-style smoke test to run outside Jupyter.
    class Markdown(str):
        pass

    def display(*objects):
        for obj in objects:
            print(obj)
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    median_absolute_error,
    roc_auc_score,
)
from sklearn.model_selection import ParameterSampler

import lightgbm as lgb
from lightgbm import LGBMClassifier, LGBMRegressor
from scipy import optimize, sparse

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 190)
pd.set_option("display.max_colwidth", None)

RANDOM_STATE = 42
NOTEBOOK_SCHEMA_VERSION = "2.2"
TARGET_ADVERTISERS = [1458, 2997]

# Data resolution
USE_KAGGLEHUB = False
KAGGLE_DATASET_SLUG = "pleaseholdme/ipinyou"
DATA_ROOT_OVERRIDE = os.getenv("IPINYOU_DATA_ROOT")

# Eligibility thresholds are data-quality gates, not statistical guarantees.
MIN_TRAIN_ROWS = 50_000
MIN_TRAIN_CLICKS = 100
MIN_TEST_ROWS = 20_000
MIN_TEST_CLICKS = 20

# Outer chronological development split.
FIT_FRAC = 0.70
CAL_FRAC = 0.15

# Hyperparameter selection happens only inside FIT_FRAC.
TUNING_WINDOW_MAX_ROWS = 600_000
TUNING_EVAL_FRAC = 0.20
CTR_TUNING_CANDIDATES = 8
COST_TUNING_CANDIDATES = 6
EARLY_STOPPING_ROUNDS = 60

# Planning and uncertainty specification.
PCTR_BINS = 10
COST_BINS = 5
UPPER_COST_TARGET_COVERAGE = 0.80
PRIMARY_BUDGET_FRACTION = 0.50
BUDGET_FRACTIONS = [0.20, 0.35, 0.50, 0.65, 0.80]

# Solver controls.
SOLVER_TIME_LIMIT_SEC = 30
SOLVER_MIP_REL_GAP = 1e-7
MAX_ACCEPTABLE_REPORTED_MIP_GAP = 1e-3

# Temporal and execution policy.
STRICT_TEMPORAL_HOLDOUT = True
ROUND_OVERLAP_CUTOFF_TO_NEXT_HOUR = True
REQUIRE_BID_TO_CLEAR = True
REQUIRE_FLOOR_TO_CLEAR = True

SMOKE_TEST = os.getenv("IPINYOU_SMOKE_TEST", "0") == "1"
if SMOKE_TEST:
    MIN_TRAIN_ROWS = 1_000
    MIN_TRAIN_CLICKS = 10
    MIN_TEST_ROWS = 500
    MIN_TEST_CLICKS = 5
    BUDGET_FRACTIONS = [0.35, 0.50, 0.65]
    TUNING_WINDOW_MAX_ROWS = 4_000
    CTR_TUNING_CANDIDATES = 2
    COST_TUNING_CANDIDATES = 2

config = pd.Series(
    {
        "schema_version": NOTEBOOK_SCHEMA_VERSION,
        "smoke_test": SMOKE_TEST,
        "target_advertisers": TARGET_ADVERTISERS,
        "fit_fraction": FIT_FRAC,
        "calibration_fraction": CAL_FRAC,
        "tuning_window_max_rows": TUNING_WINDOW_MAX_ROWS,
        "ctr_tuning_candidates": CTR_TUNING_CANDIDATES,
        "cost_tuning_candidates": COST_TUNING_CANDIDATES,
        "budget_fractions": BUDGET_FRACTIONS,
        "primary_budget_fraction": PRIMARY_BUDGET_FRACTION,
        "upper_cost_target_coverage": UPPER_COST_TARGET_COVERAGE,
        "bid_must_clear_payprice": REQUIRE_BID_TO_CLEAR,
        "bid_must_clear_slot_floor": REQUIRE_FLOOR_TO_CLEAR,
        "scipy_milp_available": hasattr(optimize, "milp"),
        "lightgbm_version": lgb.__version__,
    },
    name="value",
)
display(config.to_frame())


In [ ]:
PREPARED_DIR = ARTIFACT_ROOT / "01_prepared"
manifest_path = PREPARED_DIR / "data_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Run 01_data_integrity_and_splits.ipynb first.")
data_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
ELIGIBLE_ADVERTISERS = [int(x) for x in data_manifest["eligible_advertisers"]]
RAW = {
    advertiser: (
        pd.read_parquet(PREPARED_DIR / f"advertiser_{advertiser}_train.parquet"),
        pd.read_parquet(PREPARED_DIR / f"advertiser_{advertiser}_strict_test.parquet"),
    )
    for advertiser in ELIGIBLE_ADVERTISERS
}
print(f"Loaded prepared advertisers: {ELIGIBLE_ADVERTISERS}")


## 4. Chronological development splits

The outer split preserves the original fit/calibration/validation roles. The tuning routine then
takes a recent, capped window from the fit portion and creates another chronological
tuning-train/tuning-evaluation boundary. No holdout rows participate in model or calibrator
selection.


In [ ]:
def timestamp_group_endpoints(sorted_frame):
    """Return row boundaries that never split observations sharing one timestamp."""
    times = sorted_frame["event_time"]
    if times.isna().any():
        raise ValueError("Chronological splitting requires non-null event_time values.")
    if not times.is_monotonic_increasing:
        raise ValueError("timestamp_group_endpoints expects event_time-sorted data.")
    return np.flatnonzero(times.ne(times.shift(-1)).to_numpy()) + 1


def nearest_timestamp_boundary(endpoints, target_index, lower_exclusive=0, upper_exclusive=None):
    """Choose the valid timestamp-group boundary nearest a requested row index."""
    endpoints = np.asarray(endpoints, dtype=int)
    if upper_exclusive is None:
        upper_exclusive = int(endpoints.max())
    candidates = endpoints[
        (endpoints > int(lower_exclusive)) & (endpoints < int(upper_exclusive))
    ]
    if not len(candidates):
        raise ValueError(
            "No valid timestamp boundary exists in the requested interval; "
            "inspect timestamp parsing and unique-time counts."
        )
    return int(candidates[np.argmin(np.abs(candidates - int(target_index)))])


def chronological_split(frame, fit_frac=FIT_FRAC, cal_frac=CAL_FRAC):
    data = frame.sort_values("event_time", kind="mergesort").reset_index(drop=True)
    n_rows = len(data)
    endpoints = timestamp_group_endpoints(data)
    if n_rows < 3 or len(endpoints) < 3:
        raise ValueError("Need at least three distinct timestamps for development splits.")

    # Leave at least two timestamp groups after fit and one after calibration.
    fit_candidates = endpoints[:-2]
    fit_end = int(fit_candidates[np.argmin(np.abs(fit_candidates - int(n_rows * fit_frac)))])
    cal_end = nearest_timestamp_boundary(
        endpoints,
        int(n_rows * (fit_frac + cal_frac)),
        lower_exclusive=fit_end,
        upper_exclusive=n_rows,
    )
    fit = data.iloc[:fit_end].copy()
    calibration = data.iloc[fit_end:cal_end].copy()
    validation = data.iloc[cal_end:].copy()
    assert fit.event_time.max() < calibration.event_time.min()
    assert calibration.event_time.max() < validation.event_time.min()
    return fit, calibration, validation


def chronological_tuning_window(fit_frame, max_rows=TUNING_WINDOW_MAX_ROWS, eval_frac=TUNING_EVAL_FRAC):
    """Use a recent fit-only window and split at the nearest complete timestamp group."""
    data = fit_frame.sort_values("event_time", kind="mergesort").reset_index(drop=True)
    if len(data) > max_rows:
        # Include the whole timestamp group at the cap edge instead of truncating it.
        cutoff_time = data.iloc[-max_rows]["event_time"]
        data = data.loc[data["event_time"] >= cutoff_time].reset_index(drop=True)
    endpoints = timestamp_group_endpoints(data)
    if len(endpoints) < 2:
        raise ValueError(
            "The tuning window contains fewer than two distinct timestamps; "
            "verify millisecond timestamp parsing or increase TUNING_WINDOW_MAX_ROWS."
        )
    target = int(round(len(data) * (1 - eval_frac)))
    boundary = nearest_timestamp_boundary(
        endpoints, target, lower_exclusive=0, upper_exclusive=len(data)
    )
    tune_train = data.iloc[:boundary].copy()
    tune_eval = data.iloc[boundary:].copy()
    assert tune_train.event_time.max() < tune_eval.event_time.min()
    return tune_train, tune_eval


split_summary = []
for advertiser in ELIGIBLE_ADVERTISERS:
    fit, calibration, validation = chronological_split(RAW[advertiser][0])
    tune_train, tune_eval = chronological_tuning_window(fit)
    strict_test = RAW[advertiser][1]
    assert validation.event_time.max() < strict_test.event_time.min()
    split_summary.append(
        {
            "advertiser": advertiser,
            "fit_rows": len(fit), "fit_clicks": int(fit.click.sum()),
            "tuning_train_rows": len(tune_train),
            "tuning_eval_rows": len(tune_eval),
            "tuning_train_end": tune_train.event_time.max(),
            "tuning_eval_start": tune_eval.event_time.min(),
            "tuning_window_unique_timestamps": int(
                pd.concat([tune_train.event_time, tune_eval.event_time]).nunique()
            ),
            "calibration_rows": len(calibration),
            "calibration_clicks": int(calibration.click.sum()),
            "validation_rows": len(validation),
            "validation_clicks": int(validation.click.sum()),
            "strict_test_rows": len(strict_test),
            "strict_test_clicks": int(strict_test.click.sum()),
        }
    )
display(pd.DataFrame(split_summary))


## 5. Pre-decision features and categorical-level freezing

Only fields plausibly available when the auction opportunity arrives are used. Realized click,
clearing price, the historical bid, identifiers, and URL-like fields are excluded. Categorical
levels are learned from the relevant training portion; later unseen values are mapped to
`__OTHER__`.


In [ ]:
NUMERIC_FEATURES = [
    "slotwidth", "slotheight", "slotprice", "slot_area", "log_slotprice", "tag_count"
]
CATEGORICAL_FEATURES = [
    "weekday", "hour", "region", "city", "adexchange", "useragent",
    "slotvisibility", "slotformat", "creative",
]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FORBIDDEN_FEATURES = {
    "click", "payprice", "bidprice", "logtype", "bidid", "ipinyouid", "ip",
    "url", "urlid", "slotid", "keypage",
}
assert not (set(MODEL_FEATURES) & FORBIDDEN_FEATURES)


def engineer_features(frame):
    data = frame.copy()
    width = pd.to_numeric(data["slotwidth"], errors="coerce").fillna(0)
    height = pd.to_numeric(data["slotheight"], errors="coerce").fillna(0)
    data["slot_area"] = width * height
    data["slotprice"] = pd.to_numeric(data["slotprice"], errors="coerce").fillna(0).clip(lower=0)
    data["log_slotprice"] = np.log1p(data["slotprice"])
    if "usertag" in data:
        tags = data["usertag"].fillna("").astype(str)
        data["tag_count"] = np.where(tags.str.len().eq(0), 0, tags.str.count(",") + 1)
    else:
        data["tag_count"] = 0
    for column in NUMERIC_FEATURES:
        data[column] = pd.to_numeric(data[column], errors="coerce").fillna(0).astype(float)
    for column in CATEGORICAL_FEATURES:
        data[column] = data[column].fillna("__MISSING__").astype(str)
    data["adexchange_key"] = data["adexchange"].astype(str)
    return data


def fit_category_levels(frame):
    levels = {}
    for column in CATEGORICAL_FEATURES:
        values = set(frame[column].fillna("__MISSING__").astype(str).unique().tolist())
        values.update({"__MISSING__", "__OTHER__"})
        levels[column] = sorted(values)
    return levels


def apply_category_levels(frame, levels):
    data = frame.copy()
    for column in CATEGORICAL_FEATURES:
        values = data[column].fillna("__MISSING__").astype(str)
        allowed = set(levels[column])
        values = values.where(values.isin(allowed), "__OTHER__")
        data[column] = pd.Categorical(values, categories=levels[column])
        assert data[column].isna().sum() == 0
    return data


def deterministic_lgbm_defaults():
    """Parameters shared by every tuned/final model for reproducibility."""
    return {
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
        "subsample_freq": 1,
        "bagging_seed": RANDOM_STATE,
        "feature_fraction_seed": RANDOM_STATE,
    }


## 6. CTR hyperparameter selection and probability calibration

The original fixed LightGBM configuration is always included as a baseline. Additional sampled
configurations optimize **average precision** on the internal chronological tuning-evaluation
slice, with log loss as a tie-breaker. Average precision is appropriate for the very rare click
target, but it is always reported relative to click prevalence. Candidate models use early
stopping; the chosen structure is then refit on the entire outer fit period.

Platt scaling and isotonic regression are trained on the calibration period. The choice among
raw, Platt, and isotonic probabilities is frozen using validation log loss. The strict test period
is not consulted during either search.


In [ ]:
def expected_calibration_error(y_true, probability, n_bins=10):
    y_true = np.asarray(y_true, int)
    probability = np.asarray(probability, float)
    order = np.argsort(probability)
    bins = np.array_split(order, n_bins)
    ece, rows = 0.0, []
    for indices in bins:
        if not len(indices):
            continue
        predicted = float(np.mean(probability[indices]))
        observed = float(np.mean(y_true[indices]))
        weight = len(indices) / len(y_true)
        ece += weight * abs(predicted - observed)
        rows.append((len(indices), predicted, observed, probability[indices].min(), probability[indices].max()))
    table = pd.DataFrame(
        rows, columns=["n", "predicted", "observed", "min_probability", "max_probability"]
    )
    return float(ece), table


def probability_metrics(y_true, probability, prefix=""):
    y_true = np.asarray(y_true, int)
    probability = np.clip(np.asarray(probability, float), 1e-9, 1 - 1e-9)
    prevalence = float(np.mean(y_true))
    ap = average_precision_score(y_true, probability) if y_true.sum() > 0 else np.nan
    ece = expected_calibration_error(y_true, probability)[0]
    return {
        prefix + "prevalence": prevalence,
        prefix + "roc_auc": roc_auc_score(y_true, probability) if len(np.unique(y_true)) > 1 else np.nan,
        prefix + "average_precision": ap,
        prefix + "ap_lift_vs_prevalence": ap / prevalence if prevalence > 0 else np.nan,
        prefix + "log_loss": log_loss(y_true, probability, labels=[0, 1]),
        prefix + "brier": brier_score_loss(y_true, probability),
        prefix + "ece": ece,
        prefix + "ece_over_prevalence": ece / prevalence if prevalence > 0 else np.nan,
    }


def logit_score(probability):
    probability = np.clip(np.asarray(probability, float), 1e-6, 1 - 1e-6)
    return np.log(probability / (1 - probability)).reshape(-1, 1)


CTR_PARAMETER_SPACE = {
    "learning_rate": [0.025, 0.04, 0.06],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 8, 12],
    "min_child_samples": [80, 150, 300],
    "reg_lambda": [1.0, 3.0, 6.0],
    "reg_alpha": [0.0, 0.5, 1.0],
    "colsample_bytree": [0.75, 0.90, 1.0],
    "subsample": [0.80, 0.90, 1.0],
}

CTR_BASELINE_PARAMS = {
    "learning_rate": 0.035,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 100,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "colsample_bytree": 0.85,
    "subsample": 0.90,
}


def tune_ctr_model(fit_features):
    tune_train, tune_eval = chronological_tuning_window(fit_features)
    tune_levels = fit_category_levels(tune_train)
    train_model = apply_category_levels(tune_train, tune_levels)
    eval_model = apply_category_levels(tune_eval, tune_levels)
    sampled_candidates = list(
        ParameterSampler(
            CTR_PARAMETER_SPACE,
            n_iter=max(1, CTR_TUNING_CANDIDATES - 1),
            random_state=RANDOM_STATE,
        )
    )
    candidates = [("original_baseline", CTR_BASELINE_PARAMS)] + [
        ("sampled", candidate) for candidate in sampled_candidates
    ]
    rows = []
    for candidate_id, (candidate_source, sampled) in enumerate(candidates, 1):
        params = {
            "objective": "binary",
            "n_estimators": 1_200,
            **deterministic_lgbm_defaults(),
            **sampled,
        }
        model = LGBMClassifier(**params)
        model.fit(
            train_model[MODEL_FEATURES],
            train_model["click"].astype(int),
            categorical_feature=CATEGORICAL_FEATURES,
            eval_set=[(eval_model[MODEL_FEATURES], eval_model["click"].astype(int))],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
        probability = model.predict_proba(eval_model[MODEL_FEATURES])[:, 1]
        metrics = probability_metrics(eval_model["click"].astype(int), probability)
        rows.append(
            {
                "candidate": candidate_id,
                "candidate_source": candidate_source,
                "best_iteration": int(model.best_iteration_ or params["n_estimators"]),
                "average_precision": metrics["average_precision"],
                "roc_auc": metrics["roc_auc"],
                "log_loss": metrics["log_loss"],
                **sampled,
            }
        )
    search = pd.DataFrame(rows)
    search["ap_rank_value"] = search["average_precision"].fillna(-np.inf)
    search = search.sort_values(
        ["ap_rank_value", "log_loss"], ascending=[False, True]
    ).drop(columns="ap_rank_value").reset_index(drop=True)
    winner = search.iloc[0].to_dict()
    integer_keys = {"num_leaves", "max_depth", "min_child_samples"}
    selected = {
        key: int(winner[key]) if key in integer_keys else float(winner[key])
        for key in CTR_PARAMETER_SPACE
    }
    selected["n_estimators"] = max(50, int(winner["best_iteration"]))
    return selected, search, len(tune_train), len(tune_eval)


def fit_ctr_bundle(fit_features, calibration_features, validation_features):
    selected, search, tuning_train_rows, tuning_eval_rows = tune_ctr_model(fit_features)
    levels = fit_category_levels(fit_features)
    fit_model = apply_category_levels(fit_features, levels)
    calibration_model = apply_category_levels(calibration_features, levels)
    validation_model = apply_category_levels(validation_features, levels)
    final_params = {
        "objective": "binary",
        **deterministic_lgbm_defaults(),
        **selected,
    }
    model = LGBMClassifier(**final_params)
    model.fit(
        fit_model[MODEL_FEATURES],
        fit_model["click"].astype(int),
        categorical_feature=CATEGORICAL_FEATURES,
    )

    calibration_raw = model.predict_proba(calibration_model[MODEL_FEATURES])[:, 1]
    validation_raw = model.predict_proba(validation_model[MODEL_FEATURES])[:, 1]
    candidate_probabilities = {"raw": validation_raw}
    calibrators = {}
    if int(calibration_model.click.sum()) >= 20 and int((1 - calibration_model.click).sum()) >= 20:
        platt = LogisticRegression(C=1e6, solver="lbfgs")
        platt.fit(logit_score(calibration_raw), calibration_model.click.astype(int))
        candidate_probabilities["platt"] = platt.predict_proba(logit_score(validation_raw))[:, 1]
        calibrators["platt"] = platt
    if int(calibration_model.click.sum()) >= 100:
        isotonic = IsotonicRegression(out_of_bounds="clip")
        isotonic.fit(calibration_raw, calibration_model.click.astype(int))
        candidate_probabilities["isotonic"] = isotonic.predict(validation_raw)
        calibrators["isotonic"] = isotonic
    validation_log_loss = {
        name: log_loss(
            validation_model.click.astype(int), np.clip(probability, 1e-9, 1 - 1e-9), labels=[0, 1]
        )
        for name, probability in candidate_probabilities.items()
    }
    chosen = min(validation_log_loss, key=validation_log_loss.get)
    return {
        "model": model,
        "levels": levels,
        "chosen": chosen,
        "calibrators": calibrators,
        "validation_log_loss_by_calibrator": validation_log_loss,
        "selected_params": selected,
        "search": search,
        "tuning_train_rows": tuning_train_rows,
        "tuning_eval_rows": tuning_eval_rows,
    }


def predict_ctr(bundle, feature_frame):
    model_frame = apply_category_levels(feature_frame, bundle["levels"])
    raw = bundle["model"].predict_proba(model_frame[MODEL_FEATURES])[:, 1]
    if bundle["chosen"] == "raw":
        return raw
    if bundle["chosen"] == "platt":
        return bundle["calibrators"]["platt"].predict_proba(logit_score(raw))[:, 1]
    return bundle["calibrators"]["isotonic"].predict(raw)


## 7. Clearing-price tuning and calibrated upper costs

The central regressor estimates conditional expected clearing price. The original fixed
configuration is included as a baseline, and sampled candidate structures are ranked using a
development score combining normalized RMSE and aggregate spend bias. After
refitting on the full fit period, an aggregate scale factor is estimated on calibration data.

For conservative planning, a quantile model produces a row-level upper-price candidate. A
one-sided split-conformal residual adjustment is then learned on the calibration period:

\[
u_i=\max\{\hat c_i,\ \hat q_i+\widehat Q_{1-\alpha}(y_i-\hat q_i)\}.
\]

The validation and strict-test coverage of \(u_i\) are reported explicitly. This is an empirical
marginal row-level calibration device. It is **not** described as a guarantee that total hourly or
campaign spend will satisfy a chance constraint under temporal dependence.


In [ ]:
COST_PARAMETER_SPACE = {
    "learning_rate": [0.025, 0.04, 0.06],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 8, 12],
    "min_child_samples": [80, 150, 300],
    "reg_lambda": [1.0, 3.0, 6.0],
    "reg_alpha": [0.0, 0.5, 1.0],
    "colsample_bytree": [0.75, 0.90, 1.0],
    "subsample": [0.80, 0.90, 1.0],
}

COST_BASELINE_PARAMS = {
    "learning_rate": 0.04,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 100,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "colsample_bytree": 0.85,
    "subsample": 0.90,
}


def tune_cost_model(fit_features):
    tune_train, tune_eval = chronological_tuning_window(fit_features)
    tune_levels = fit_category_levels(tune_train)
    train_model = apply_category_levels(tune_train, tune_levels)
    eval_model = apply_category_levels(tune_eval, tune_levels)
    sampled_candidates = list(
        ParameterSampler(
            COST_PARAMETER_SPACE,
            n_iter=max(1, COST_TUNING_CANDIDATES - 1),
            random_state=RANDOM_STATE + 17,
        )
    )
    candidates = [("original_baseline", COST_BASELINE_PARAMS)] + [
        ("sampled", candidate) for candidate in sampled_candidates
    ]
    rows = []
    y_eval = eval_model.payprice.astype(float).to_numpy()
    mean_cost = max(float(np.mean(y_eval)), 1e-9)
    for candidate_id, (candidate_source, sampled) in enumerate(candidates, 1):
        params = {
            "objective": "regression_l2",
            "n_estimators": 1_200,
            **deterministic_lgbm_defaults(),
            **sampled,
        }
        model = LGBMRegressor(**params)
        model.fit(
            train_model[MODEL_FEATURES],
            train_model.payprice.astype(float),
            categorical_feature=CATEGORICAL_FEATURES,
            eval_set=[(eval_model[MODEL_FEATURES], eval_model.payprice.astype(float))],
            eval_metric="rmse",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
        prediction = np.clip(model.predict(eval_model[MODEL_FEATURES]), 1e-3, None)
        rmse = float(np.sqrt(np.mean((y_eval - prediction) ** 2)))
        mae = float(mean_absolute_error(y_eval, prediction))
        spend_ratio = float(prediction.sum() / y_eval.sum()) if y_eval.sum() else np.nan
        score = rmse / mean_cost + 0.50 * abs(np.log(max(spend_ratio, 1e-9)))
        rows.append(
            {
                "candidate": candidate_id,
                "candidate_source": candidate_source,
                "best_iteration": int(model.best_iteration_ or params["n_estimators"]),
                "normalized_rmse_bias_score": score,
                "rmse": rmse,
                "mae": mae,
                "predicted_to_actual_spend": spend_ratio,
                **sampled,
            }
        )
    search = pd.DataFrame(rows).sort_values(
        ["normalized_rmse_bias_score", "mae"], ascending=[True, True]
    ).reset_index(drop=True)
    winner = search.iloc[0].to_dict()
    integer_keys = {"num_leaves", "max_depth", "min_child_samples"}
    selected = {
        key: int(winner[key]) if key in integer_keys else float(winner[key])
        for key in COST_PARAMETER_SPACE
    }
    selected["n_estimators"] = max(50, int(winner["best_iteration"]))
    return selected, search, len(tune_train), len(tune_eval)


def one_sided_conformal_adjustment(actual, upper_candidate, target_coverage):
    """Finite-sample higher empirical quantile for one-sided split calibration."""
    actual = np.asarray(actual, float)
    upper_candidate = np.asarray(upper_candidate, float)
    residual = actual - upper_candidate
    n_rows = len(residual)
    if n_rows == 0:
        raise ValueError("Calibration data are required for upper-cost adjustment.")
    level = min(1.0, np.ceil((n_rows + 1) * target_coverage) / n_rows)
    return float(np.quantile(residual, level, method="higher")), float(level)


def fit_cost_bundle(fit_features, calibration_features, validation_features, ctr_levels):
    selected, search, tuning_train_rows, tuning_eval_rows = tune_cost_model(fit_features)
    fit_model = apply_category_levels(fit_features, ctr_levels)
    calibration_model = apply_category_levels(calibration_features, ctr_levels)
    validation_model = apply_category_levels(validation_features, ctr_levels)
    central_params = {
        "objective": "regression_l2",
        **deterministic_lgbm_defaults(),
        **selected,
    }
    central_model = LGBMRegressor(**central_params)
    central_model.fit(
        fit_model[MODEL_FEATURES], fit_model.payprice.astype(float),
        categorical_feature=CATEGORICAL_FEATURES,
    )
    quantile_params = {
        **central_params,
        "objective": "quantile",
        "alpha": UPPER_COST_TARGET_COVERAGE,
    }
    quantile_model = LGBMRegressor(**quantile_params)
    quantile_model.fit(
        fit_model[MODEL_FEATURES], fit_model.payprice.astype(float),
        categorical_feature=CATEGORICAL_FEATURES,
    )

    calibration_central_raw = np.clip(
        central_model.predict(calibration_model[MODEL_FEATURES]), 1e-3, None
    )
    scale = float(
        calibration_model.payprice.sum() / calibration_central_raw.sum()
    ) if calibration_central_raw.sum() > 0 else 1.0
    calibration_quantile = np.clip(
        quantile_model.predict(calibration_model[MODEL_FEATURES]), 1e-3, None
    )
    adjustment, conformal_level = one_sided_conformal_adjustment(
        calibration_model.payprice.astype(float).to_numpy(),
        calibration_quantile,
        UPPER_COST_TARGET_COVERAGE,
    )

    validation_central = np.clip(
        central_model.predict(validation_model[MODEL_FEATURES]), 1e-3, None
    ) * scale
    validation_quantile = np.clip(
        quantile_model.predict(validation_model[MODEL_FEATURES]), 1e-3, None
    )
    validation_upper = np.maximum(
        validation_central, np.clip(validation_quantile + adjustment, 1e-3, None)
    )
    y_validation = validation_model.payprice.astype(float).to_numpy()
    metrics = {
        "cost_val_mae": mean_absolute_error(y_validation, validation_central),
        "cost_val_median_ae": median_absolute_error(y_validation, validation_central),
        "cost_val_pred_to_actual_spend": (
            float(validation_central.sum() / y_validation.sum()) if y_validation.sum() else np.nan
        ),
        "upper_cost_target_coverage": UPPER_COST_TARGET_COVERAGE,
        "upper_cost_validation_coverage": float(np.mean(y_validation <= validation_upper)),
        "upper_cost_validation_mean_width": float(np.mean(validation_upper - validation_central)),
        "aggregate_scale_factor": scale,
        "conformal_adjustment_native": adjustment,
        "conformal_quantile_level": conformal_level,
    }
    return {
        "central_model": central_model,
        "quantile_model": quantile_model,
        "scale": scale,
        "adjustment": adjustment,
        "levels": ctr_levels,
        "metrics": metrics,
        "selected_params": selected,
        "search": search,
        "tuning_train_rows": tuning_train_rows,
        "tuning_eval_rows": tuning_eval_rows,
    }


def predict_cost(bundle, feature_frame):
    model_frame = apply_category_levels(feature_frame, bundle["levels"])
    central = np.clip(
        bundle["central_model"].predict(model_frame[MODEL_FEATURES]), 1e-3, None
    ) * bundle["scale"]
    quantile = np.clip(
        bundle["quantile_model"].predict(model_frame[MODEL_FEATURES]), 1e-3, None
    )
    upper = np.maximum(
        central, np.clip(quantile + bundle["adjustment"], 1e-3, None)
    )
    return central, upper, quantile


## 8. Frozen planning segments and out-of-fit coefficients

Planning groups are

\[
g=(\text{hour},\ \text{ad exchange},\ \text{pCTR bucket},\ \text{cost bucket}).
\]

Bucket edges and group value/cost coefficients come from the calibration-plus-validation
reference period, whose base-model predictions are out of fit. Full pre-holdout history is still
used to estimate capacity. When a capacity group is absent from the reference period, the code
uses a declared fallback coefficient and reports the fallback rate.

The scenario budget is based on predicted group cost and historical capacity, projected over the
holdout calendar. It remains an experimental cap; utilization diagnostics determine whether it
was actually binding.


In [ ]:
def frozen_edges(values, n_bins):
    values = np.asarray(values, float)
    edges = np.unique(np.quantile(values, np.linspace(0, 1, n_bins + 1)))
    if len(edges) < 2:
        edges = np.array([np.nanmin(values), np.nanmax(values) + 1e-9])
    return edges


def assign_bucket(values, edges):
    internal = np.asarray(edges[1:-1], float)
    return np.searchsorted(internal, np.asarray(values, float), side="right").astype(int)


def score_frame(raw_frame, ctr_bundle, cost_bundle):
    scored = engineer_features(raw_frame)
    scored["pctr"] = predict_ctr(ctr_bundle, scored)
    central, upper, quantile = predict_cost(cost_bundle, scored)
    scored["central_cost_pred"] = central
    scored["upper_cost_pred"] = upper
    scored["quantile_cost_pred"] = quantile
    return scored


def add_planning_keys(frame, pctr_edges, cost_edges):
    data = frame.copy()
    data["pctr_bucket"] = assign_bucket(data.pctr, pctr_edges)
    data["cost_bucket"] = assign_bucket(data.central_cost_pred, cost_edges)
    data["hour_key"] = data["event_time"].dt.hour.astype(int).astype(str)
    data["adexchange_key"] = data["adexchange"].astype(str)
    data["day"] = data.event_time.dt.date
    data["slot_key"] = data.event_time.dt.floor("h")
    return data


def build_scored_frames(train_raw, reference_raw, test_raw, ctr_bundle, cost_bundle):
    train_scored = score_frame(train_raw, ctr_bundle, cost_bundle)
    reference_scored = score_frame(reference_raw, ctr_bundle, cost_bundle)
    test_scored = score_frame(test_raw, ctr_bundle, cost_bundle)
    pctr_edges = frozen_edges(reference_scored.pctr, PCTR_BINS)
    cost_edges = frozen_edges(reference_scored.central_cost_pred, COST_BINS)
    train_scored = add_planning_keys(train_scored, pctr_edges, cost_edges)
    reference_scored = add_planning_keys(reference_scored, pctr_edges, cost_edges)
    test_scored = add_planning_keys(test_scored, pctr_edges, cost_edges)
    return train_scored, reference_scored, test_scored, pctr_edges, cost_edges


GROUP_COLS = ["hour_key", "adexchange_key", "pctr_bucket", "cost_bucket"]


def hour_exposure_counts(scored_frame):
    return scored_frame.groupby("hour_key", observed=True)["day"].nunique().astype(int).to_dict()


def build_planning_table(capacity_scored, coefficient_scored):
    """Use all earlier data for capacity and out-of-fit reference data for coefficients."""
    exposures = hour_exposure_counts(capacity_scored)
    capacity = (
        capacity_scored.groupby(GROUP_COLS, observed=True)
        .agg(train_count=("click", "size"))
        .reset_index()
    )
    fallback = (
        capacity_scored.groupby(GROUP_COLS, observed=True)
        .agg(
            fallback_expected_ctr=("pctr", "mean"),
            fallback_central_cost=("central_cost_pred", "mean"),
            fallback_upper_cost=("upper_cost_pred", "mean"),
        )
        .reset_index()
    )
    reference = (
        coefficient_scored.groupby(GROUP_COLS, observed=True)
        .agg(
            reference_rows=("click", "size"),
            reference_expected_ctr=("pctr", "mean"),
            reference_central_cost=("central_cost_pred", "mean"),
            reference_upper_cost=("upper_cost_pred", "mean"),
            observed_reference_ctr=("click", "mean"),
        )
        .reset_index()
    )
    planning = capacity.merge(fallback, on=GROUP_COLS, how="left").merge(
        reference, on=GROUP_COLS, how="left"
    )
    planning["coefficient_fallback"] = planning["reference_rows"].isna()
    planning["expected_ctr"] = planning["reference_expected_ctr"].fillna(
        planning["fallback_expected_ctr"]
    )
    planning["central_cost"] = planning["reference_central_cost"].fillna(
        planning["fallback_central_cost"]
    )
    planning["upper_cost"] = planning["reference_upper_cost"].fillna(
        planning["fallback_upper_cost"]
    )
    planning["hour_exposures"] = planning["hour_key"].map(exposures).astype(int)
    planning["capacity_per_exposure"] = planning["train_count"] / planning["hour_exposures"]
    planning["forecast_capacity"] = 0
    for _, indices in planning.groupby("hour_key", observed=True).groups.items():
        indices = list(indices)
        expected = planning.loc[indices, "capacity_per_exposure"].to_numpy(float)
        base = np.floor(expected).astype(int)
        target = int(round(expected.sum()))
        extras = max(0, target - int(base.sum()))
        if extras:
            order = np.argsort(-(expected - base))
            base[order[:extras]] += 1
        planning.loc[indices, "forecast_capacity"] = base
    planning["forecast_capacity"] = planning["forecast_capacity"].astype(int)
    planning = planning[planning.forecast_capacity > 0].copy().reset_index(drop=True)
    planning["group_id"] = np.arange(len(planning))
    planning["unit_value"] = 1.0
    return planning


def calendar_hour_slots(holdout_frame):
    start = holdout_frame.event_time.min().floor("h")
    end = holdout_frame.event_time.max().floor("h")
    return list(pd.date_range(start, end, freq="h"))


def projected_full_spend_reference(planning, holdout_frame):
    """Project central predicted cost from earlier capacity over the holdout calendar."""
    by_hour = (
        planning.assign(
            forecast_spend=lambda data: data.central_cost * data.forecast_capacity
        )
        .groupby("hour_key", observed=True).forecast_spend.sum()
        .to_dict()
    )
    projected = sum(float(by_hour.get(str(timestamp.hour), 0.0)) for timestamp in calendar_hour_slots(holdout_frame))
    if not projected > 0:
        raise ValueError("Projected full-spend reference must be positive.")
    return float(projected)


## Distribution-shift diagnostic

PSI is descriptive here. It is not used to choose a model after seeing strict test.


In [ ]:
def population_stability_index(train_values, test_values, bins=10):
    train_values = np.asarray(train_values, float)
    test_values = np.asarray(test_values, float)
    edges = np.unique(np.quantile(train_values, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return np.nan
    edges[0], edges[-1] = -np.inf, np.inf
    train_hist = np.histogram(train_values, bins=edges)[0].astype(float)
    test_hist = np.histogram(test_values, bins=edges)[0].astype(float)
    train_share = np.clip(train_hist / train_hist.sum(), 1e-6, None)
    test_share = np.clip(test_hist / test_hist.sum(), 1e-6, None)
    return float(np.sum((test_share - train_share) * np.log(test_share / train_share)))


## Execute and persist frozen model outputs

This is the expensive stage. Each advertiser is modeled independently, consistent with
the paper's evidence that advertiser response patterns differ materially.


In [ ]:
import joblib

MODEL_DIR = ARTIFACT_ROOT / "02_models_and_scores"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_rows = []
model_manifest = {
    "schema_version": NOTEBOOK_SCHEMA_VERSION,
    "eligible_advertisers": ELIGIBLE_ADVERTISERS,
    "full_spend_reference_native": {},
    "pctr_edges": {},
    "cost_edges": {},
}

for run_number, advertiser in enumerate(ELIGIBLE_ADVERTISERS, 1):
    start_time = time.time()
    print(f"[{run_number}/{len(ELIGIBLE_ADVERTISERS)}] tuning advertiser {advertiser}")
    train_raw, test_raw = RAW[advertiser]
    assert train_raw.event_time.max() < test_raw.event_time.min()
    fit, calibration, validation = chronological_split(train_raw)
    fit_features, calibration_features, validation_features = [
        engineer_features(frame) for frame in (fit, calibration, validation)
    ]

    # Hyperparameters use only the fit-period internal chronology. Calibration and validation
    # retain separate roles, and the strict test is untouched until final scoring.
    ctr_bundle = fit_ctr_bundle(fit_features, calibration_features, validation_features)
    cost_bundle = fit_cost_bundle(
        fit_features, calibration_features, validation_features, ctr_bundle["levels"]
    )
    validation_probability = predict_ctr(ctr_bundle, validation_features)
    validation_metrics = probability_metrics(
        validation_features.click.astype(int), validation_probability, "val_"
    )

    reference_raw = pd.concat([calibration, validation], ignore_index=True).sort_values(
        "event_time", kind="mergesort"
    )
    train_scored, reference_scored, test_scored, pctr_edges, cost_edges = build_scored_frames(
        train_raw, reference_raw, test_raw, ctr_bundle, cost_bundle
    )
    test_metrics = probability_metrics(test_scored.click.astype(int), test_scored.pctr, "test_")
    test_calibration_table = expected_calibration_error(
        test_scored.click.astype(int), test_scored.pctr
    )[1]
    test_calibration_table.insert(0, "advertiser", advertiser)

    y_test_cost = test_scored.payprice.astype(float).to_numpy()
    test_cost_metrics = {
        "cost_test_mae": mean_absolute_error(y_test_cost, test_scored.central_cost_pred),
        "cost_test_pred_to_actual_spend": (
            float(test_scored.central_cost_pred.sum() / y_test_cost.sum())
            if y_test_cost.sum() else np.nan
        ),
        "upper_cost_test_coverage": float(np.mean(y_test_cost <= test_scored.upper_cost_pred)),
        "upper_cost_test_mean_width": float(
            np.mean(test_scored.upper_cost_pred - test_scored.central_cost_pred)
        ),
    }
    drift = {
        "pctr_psi": population_stability_index(reference_scored.pctr, test_scored.pctr),
        "cost_pred_psi": population_stability_index(
            reference_scored.central_cost_pred, test_scored.central_cost_pred
        ),
    }
    planning = build_planning_table(train_scored, reference_scored)
    full_spend_reference = projected_full_spend_reference(planning, test_scored)

    model_row = {
        "advertiser": advertiser,
        "ctr_calibration_choice": ctr_bundle["chosen"],
        **validation_metrics, **test_metrics,
        **cost_bundle["metrics"], **test_cost_metrics, **drift,
        "planning_coefficient_fallback_rate": float(planning.coefficient_fallback.mean()),
        "train_rows": len(train_raw), "strict_test_rows": len(test_raw),
        "strict_test_days": int(test_scored.day.nunique()),
        "full_spend_reference_native": full_spend_reference,
        "full_spend_reference_rmb": full_spend_reference / 1000,
        "strict_temporal_gap_seconds": float(
            (test_raw.event_time.min() - train_raw.event_time.max()).total_seconds()
        ),
        "ctr_selected_params": json.dumps(
            ctr_bundle["selected_params"], sort_keys=True, default=float
        ),
        "cost_selected_params": json.dumps(
            cost_bundle["selected_params"], sort_keys=True, default=float
        ),
        "runtime_sec": time.time() - start_time,
    }
    model_rows.append(model_row)

    # Persist only what later stages need: models, compact planning coefficients, and scored test.
    joblib.dump(
        {"ctr_bundle": ctr_bundle, "cost_bundle": cost_bundle},
        MODEL_DIR / f"advertiser_{advertiser}_models.joblib",
        compress=3,
    )
    planning.to_parquet(
        MODEL_DIR / f"advertiser_{advertiser}_planning.parquet",
        index=False, compression="zstd",
    )
    test_scored.to_parquet(
        MODEL_DIR / f"advertiser_{advertiser}_test_scored.parquet",
        index=False, compression="zstd",
    )
    ctr_bundle["search"].to_csv(MODEL_DIR / f"advertiser_{advertiser}_ctr_search.csv", index=False)
    cost_bundle["search"].to_csv(MODEL_DIR / f"advertiser_{advertiser}_cost_search.csv", index=False)
    test_calibration_table.to_csv(
        MODEL_DIR / f"advertiser_{advertiser}_test_calibration.csv", index=False
    )
    model_manifest["full_spend_reference_native"][str(advertiser)] = float(full_spend_reference)
    model_manifest["pctr_edges"][str(advertiser)] = np.asarray(pctr_edges, float).tolist()
    model_manifest["cost_edges"][str(advertiser)] = np.asarray(cost_edges, float).tolist()

model_metrics = pd.DataFrame(model_rows).sort_values("advertiser")
model_metrics.to_csv(MODEL_DIR / "model_metrics.csv", index=False)
(MODEL_DIR / "model_manifest.json").write_text(
    json.dumps(model_manifest, indent=2), encoding="utf-8"
)
display(model_metrics)
print(f"Model artifacts written to {MODEL_DIR}")
